# VERGIL RL Training via GRPO (Google Colab)

This notebook trains a 0.5B parameter LLM (Qwen2.5) to act as the VERGIL agent using **Group Relative Policy Optimization (GRPO)** and **Unsloth** for 4-bit memory efficient training.

**Hardware Requirements:** T4 GPU (free tier Colab is sufficient).

## 1. Setup Environment

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install gymnasium networkx datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-4_vjqewa/unsloth_8822e5813119400da2aed3872b8ea7a0
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-4_vjqewa/unsloth_8822e5813119400da2aed3872b8ea7a0
  Resolved https://github.com/unslothai/unsloth.git to commit b09aa82a3ac9ac9794c497e4b8f747f77e52b162
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 138.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 

In [ ]:
# Clone the VERGIL repository to access the environment and CDG engine
!rm -rf Vergil
!git clone https://github.com/laksh718/Vergil.git
%cd Vergil

import sys
sys.path.insert(0, '.')

## 2. Load VERGIL Engine and Define Reward

In [ ]:
from vergil.core.env import VERGILEnv
from vergil.core.types import AgentAction, ActionType, CommitmentStatus
from vergil.core.pomdp import POMDPWrapper
from vergil.curriculum.scenario_generator import ScenarioGenerator
from vergil.curriculum.curriculum_engine import CurriculumEngine
from vergil.curriculum.failure_db import FailureTopologyDatabase
from scripts.train_grpo_colab import state_to_prompt, parse_llm_output, vergil_reward_function, simulate_task_progress

In [ ]:
print("\n🌍 Initializing VERGIL environment...")
env = VERGILEnv(seed=42, config={'max_steps_per_episode': 20, 'step_hours': 2})
pomdp = POMDPWrapper(env)

failure_db = FailureTopologyDatabase(db_path='/tmp/vergil_ftd_grpo.sqlite')
scenario_gen = ScenarioGenerator(seed=42)
curriculum = CurriculumEngine(
    failure_db=failure_db, scenario_generator=scenario_gen, initial_stage=1
)

def reward_fn(prompts, completions, **kw):
    # Wrap the environment specifically for TRL GRPO callback
    return vergil_reward_function(prompts, completions, env=env, pomdp=pomdp)

## 3. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True, # 4-bit quantization helps fit on T4
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16, # Rank of the LoRA matrices
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f"\nLoRA adapter added. Trainable params: {model.num_parameters(only_trainable=True):,}")

## 4. Generate Training Dataset
We need to randomly sample some states from the VERGIL engine to generate text prompts.

In [ ]:
training_prompts = []

for i in range(200):
    env.curriculum_stage = min(2, curriculum.current_stage)
    scenario = curriculum.generate_next_episode()
    state, belief, info = pomdp.reset(scenario=scenario)
    
    for j in range(min(5, env._max_steps)):
        simulate_task_progress(env)
        prompt = state_to_prompt(state, env)
        training_prompts.append(prompt)
        
        # Random action to step environment forward
        pending = [n for n in state.cdg_nodes if n.status == CommitmentStatus.PENDING]
        action = AgentAction(ActionType.ACCEPT, target_node_id=pending[0].node_id) if pending else AgentAction(ActionType.DO_NOTHING)
        
        state, belief, reward, term, trunc, step_info = pomdp.step(action)
        simulate_task_progress(env)
        if term or trunc: break

from datasets import Dataset
dataset = Dataset.from_dict({"prompt": training_prompts[:500]})
print(f"Generated {len(dataset)} training prompts.")

## 5. Train with GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

training_config = GRPOConfig(
    output_dir="/tmp/vergil_grpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    max_completion_length=256,
    num_generations=4, # Rollout generation per prompt (GRPO baseline)
    logging_steps=10,
    save_steps=100,
    warmup_steps=20,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    args=training_config,
    train_dataset=dataset,
    reward_funcs=[reward_fn],
    processing_class=tokenizer,
)

print("🚀 Starting GRPO Training...")
trainer.train()

## 6. Save Model to HuggingFace

In [ ]:
!huggingface-cli login --token 'token placement'\n

In [ ]:
# Push the LoRA fine-tuned model
# Remember to replace 'yourusername' with your actual HF username
HF_REPO_NAME = "yourusername/vergil-qwen-grpo"

model.push_to_hub(HF_REPO_NAME)
tokenizer.push_to_hub(HF_REPO_NAME)
print(f"\n✅ Successfully pushed model to Hugging Face Hub!")